In [ ]:
import pandas as pd

In [ ]:
path = "data/olist/" # caminho raiz até os arquivos *.csv
orders = pd.read_csv(path + "olist_orders_dataset.csv")

In [ ]:
orders

In [ ]:
colunas = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders[colunas] = orders[colunas].apply(pd.to_datetime)
orders

In [ ]:
orders['order_status'].describe()

In [ ]:
agrupado = orders.groupby('order_status').size()
agrupado

In [ ]:
### TEMPO DE ENTREGA REALIZADO - ESTIMADO ###
entregues = orders.loc[orders["order_status"] == "delivered"].copy()

entregues["delivered_in_days"] = (
    entregues["order_estimated_delivery_date"]
    - entregues["order_delivered_customer_date"]
).dt.days

entregues['delivered_in_days'].describe()

In [ ]:
mins = entregues.loc[entregues['delivered_in_days'] < 0]
med = entregues[(entregues['delivered_in_days'] > 0) & (entregues['delivered_in_days'] < 12)]
a=(entregues["delivered_in_days"] < 0).mean() * 100
a
num_pedidos = len(entregues)
num_pedidos_antes = len(mins)
num_pedidos_prazo = len(med)
print (f'{num_pedidos_antes/num_pedidos * 100} % de pedidos entregues antes do prazo')
print (f'{num_pedidos_prazo/num_pedidos * 100} % de pedidos entregues no prazo')
print(f'{(num_pedidos - (num_pedidos_antes + num_pedidos_prazo))/num_pedidos * 100} % de pedidos atrasados')

In [ ]:
import plotly.express as px

fig = px.histogram(
    entregues,
    x="delivered_in_days",  # atraso em dias
    nbins=100,
    title="Tempo de entrega dos pedidos em dias (Estimado - Realizado)"
)
fig.show()

In [ ]:
top_10_atrasos = entregues['delivered_in_days'].to_frame()
top_10_atrasos.describe()

In [ ]:
### PEDIDOS NÃO ENTREGUES ###
nao_entregues = orders[orders['order_status'] != "delivered"]
nao_entregues

In [ ]:
### MAIORES 20 CLIENTES  ###
top_10_custormers = orders['customer_id'].value_counts()[:20]
top_10_custormers

In [ ]:
agrupado = orders['customer_id'].describe()
agrupado

In [ ]:
agrupado = orders['order_purchase_timestamp'].describe()
agrupado

In [ ]:
orders_per_month_year = (
    orders
    .assign(
        ano=orders["order_purchase_timestamp"].dt.year,
        mes=orders["order_purchase_timestamp"].dt.month
    )
    .groupby(["ano", "mes"], as_index=False)
    .size()
    .rename(columns={"size": "quantidade"})
)

orders_per_month_year

In [ ]:
# datetime mensal
orders_per_month_year["data"] = pd.to_datetime(
    orders_per_month_year["ano"].astype(str) + "-" +
    orders_per_month_year["mes"].astype(str) + "-01"
)

# agregado por ano
por_ano = (
    orders_per_month_year
    .groupby("ano", as_index=False)["quantidade"]
    .sum()
)

# agregado por mês (todos os anos juntos)
por_mes = (
    orders_per_month_year
    .groupby("mes", as_index=False)["quantidade"]
    .sum()
)

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Evolução Mensal",
        "Total por Ano",
        "Distribuição por Mês"
    ]
)

# 1️⃣ Evolução mensal
fig.add_trace(
    go.Scatter(
        x=orders_per_month_year["data"],
        y=orders_per_month_year["quantidade"],
        mode="lines+markers",
        name="Mensal"
    ),
    row=1, col=1
)

# 2️⃣ Total por ano
fig.add_trace(
    go.Bar(
        x=por_ano["ano"],
        y=por_ano["quantidade"],
        name="Ano"
    ),
    row=1, col=2
)

# 3️⃣ Distribuição por mês
fig.add_trace(
    go.Bar(
        x=por_mes["mes"],
        y=por_mes["quantidade"],
        name="Mês"
    ),
    row=1, col=3
)

fig.update_layout(
    height=450,
    showlegend=False,
    title_text="Análise Temporal de Pedidos",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()